In [1]:
# ---- Cell 1 code ----
# Environment & path config (one CSV per table)
import pandas as pd
import numpy as np
from pathlib import Path

# Raw string stops Windows backslashes being read as escape characters
DATA_DIR = Path(r"C:\Users\majid\Desktop\Data Analyst Portfolio\3_SaaS_Churn_Analytics\Data")

EXPECTED_TABLES = ['dim_customers', 'dim_plans', 'fact_subscriptions', 'fact_payments', 'fact_usage', 'fact_support']

# Fail fast with a readable message instead of a cryptic pandas traceback later
assert DATA_DIR.exists(), f"Data directory not found: {DATA_DIR}"

# Map snake_case file names to paths; skip our own exports so re-runs never re-ingest them
file_map = {
    f.stem.strip().lower().replace(' ', '_'): f
    for f in sorted(DATA_DIR.glob('*.csv'))
    if not f.name.startswith('cleaned_')
}

print(f"--- Source folder: {DATA_DIR.name} ---")
for name, path in file_map.items():
    print(f"  '{path.name}' -> '{name}' | {path.stat().st_size:,} bytes")

missing = [t for t in EXPECTED_TABLES if t not in file_map]
print(f"Missing expected tables: {missing if missing else 'None'}")

# Stop here if the folder does not match the schema map
assert not missing, f"Missing required tables: {missing}"
print(f"Tables found: {len(file_map)}")

--- Source folder: Data ---
  'dim_customers.csv' -> 'dim_customers' | 77,750 bytes
  'dim_plans.csv' -> 'dim_plans' | 130 bytes
  'fact_payments.csv' -> 'fact_payments' | 160,340 bytes
  'fact_subscriptions.csv' -> 'fact_subscriptions' | 76,501 bytes
  'fact_support.csv' -> 'fact_support' | 48,071 bytes
  'fact_usage.csv' -> 'fact_usage' | 1,303,678 bytes
Missing expected tables: None
Tables found: 6


In [2]:
# ---- Cell 2 code ----
# Raw ingestion: one dictionary holds every table
raw_tables = {}

for name in EXPECTED_TABLES:
    # Read as-is; type cleaning happens later so each step stays testable
    raw_tables[name] = pd.read_csv(file_map[name])

# Baseline data quality: shape, null counts and dtypes per table
for name, df in raw_tables.items():
    print(f"--- {name} | shape: {df.shape} ---")
    print(df.isna().sum()[df.isna().sum() > 0])  # show only columns that have nulls
    print(df.dtypes.to_dict())

# Gate: every table must contain data before we transform anything
for name, df in raw_tables.items():
    assert len(df) > 0, f"{name}: table is empty"

print(f"Tables loaded: {len(raw_tables)} | Total rows: {sum(len(d) for d in raw_tables.values()):,}")

--- dim_customers | shape: (1550, 6) ---
country     348
industry    245
dtype: int64
{'customer_id': dtype('O'), 'signup_date': dtype('O'), 'country': dtype('O'), 'industry': dtype('O'), 'company_size': dtype('O'), 'segment': dtype('O')}
--- dim_plans | shape: (3, 4) ---
Series([], dtype: int64)
{'plan_id': dtype('O'), 'plan_name': dtype('O'), 'monthly_price': dtype('float64'), 'billing_cycle': dtype('O')}
--- fact_subscriptions | shape: (1500, 6) ---
end_date    1043
dtype: int64
{'subscription_id': dtype('O'), 'customer_id': dtype('O'), 'plan_id': dtype('O'), 'start_date': dtype('O'), 'end_date': dtype('O'), 'status': dtype('O')}
--- fact_payments | shape: (3063, 6) ---
Series([], dtype: int64)
{'payment_id': dtype('O'), 'subscription_id': dtype('O'), 'payment_date': dtype('O'), 'amount': dtype('float64'), 'payment_status': dtype('O'), 'payment_method': dtype('O')}
--- fact_usage | shape: (24756, 6) ---
feature_name    942
dtype: int64
{'usage_id': dtype('O'), 'customer_id': dtype('

In [3]:
# ---- Cell 3 code ----
# Header normalization (snake_case) and ID standardization
for df in raw_tables.values():
    # Consistent headers so join keys match across tables
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

for name, df in raw_tables.items():
    # Treat every column ending in _id as a key, so no table needs a hardcoded list
    id_cols = [c for c in df.columns if c.endswith('_id')]
    for col in id_cols:
        # Cast to text, trim spaces, drop float artifacts like "101.0"
        df[col] = df[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
        df[col] = df[col].replace('nan', np.nan)  # restore true nulls for later checks
    print(f"--- {name} | shape: {df.shape} ---")
    print({c: str(df[c].dtype) for c in id_cols})

# Gate: no spaces, uppercase, or duplicate headers may remain
for name, df in raw_tables.items():
    assert all(c == c.lower() and ' ' not in c for c in df.columns), f"{name}: headers not snake_case"
    assert df.columns.is_unique, f"{name}: duplicate headers"

print(f"Columns per table: { {n: d.shape[1] for n, d in raw_tables.items()} }")

--- dim_customers | shape: (1550, 6) ---
{'customer_id': 'object'}
--- dim_plans | shape: (3, 4) ---
{'plan_id': 'object'}
--- fact_subscriptions | shape: (1500, 6) ---
{'subscription_id': 'object', 'customer_id': 'object', 'plan_id': 'object'}
--- fact_payments | shape: (3063, 6) ---
{'payment_id': 'object', 'subscription_id': 'object'}
--- fact_usage | shape: (24756, 6) ---
{'usage_id': 'object', 'customer_id': 'object'}
--- fact_support | shape: (784, 7) ---
{'ticket_id': 'object', 'customer_id': 'object'}
Columns per table: {'dim_customers': 6, 'dim_plans': 4, 'fact_subscriptions': 6, 'fact_payments': 6, 'fact_usage': 6, 'fact_support': 7}


In [4]:
# ---- Cell 4 code ----
# SaaS quality gates
# Blanket non-negative rule covers plan money; payment amounts get their own refund-aware rule
FINANCIAL_COLS = ['mrr', 'price']
PRIMARY_KEYS = {'dim_customers': 'customer_id', 'dim_plans': 'plan_id',
                'fact_subscriptions': 'subscription_id', 'fact_payments': 'payment_id'}
STATUS_FIXES = {'canceled': 'cancelled', 'act': 'active', 'active_plan': 'active'}  # one label per real status

def clean_table(df):
    df = df.copy()  # work on a copy so the raw data stays untouched

    # Convert date columns first, so they are not filled with text later
    for col in [c for c in df.columns if 'date' in c or 'timestamp' in c]:
        df[col] = pd.to_datetime(df[col], errors='coerce')  # bad values become NaT

    num_cols = df.select_dtypes(include='number').columns
    txt_cols = df.select_dtypes(include='object').columns

    # Labels arrive as 'paid' and 'PAID', so standardise case before any grouping
    cat_cols = [c for c in txt_cols if any(k in c for k in ['status', 'method', 'type'])]
    for col in cat_cols:
        df[col] = df[col].str.strip().str.lower()

    # Spelling variants of one status would split active and churn counts
    for col in [c for c in cat_cols if 'status' in c]:
        df[col] = df[col].replace(STATUS_FIXES)

    # Plan IDs arrive as 'P_PRO' and 'p_pro', which would break joins to dim_plans
    if 'plan_id' in df.columns:
        df['plan_id'] = df['plan_id'].str.strip().str.lower()

    # Money nulls stay null: an unknown amount is not a free one
    fill_num = [c for c in num_cols if c not in FINANCIAL_COLS + ['amount']]
    df[fill_num] = df[fill_num].fillna(0)                # missing counts become 0
    df[txt_cols] = df[txt_cols].fillna('Not Specified')  # missing text becomes a label
    return df

tables = {name: clean_table(df) for name, df in raw_tables.items()}

# Customers: drop exact copies, then keep the latest record per repeated ID
cust = tables['dim_customers']
rows_before = len(cust)
cust = cust.drop_duplicates()  # exact copies carry no extra information
exact_dropped = rows_before - len(cust)

dup_mask = cust.duplicated(subset='customer_id', keep='last')  # older versions of a repeated ID
quarantine_customers = cust[dup_mask].copy()  # keep dropped rows so nothing disappears silently
tables['dim_customers'] = cust[~dup_mask].reset_index(drop=True)

print(f"--- dim_customers | exact duplicates dropped: {exact_dropped:,} | older records quarantined: {len(quarantine_customers):,} ---")
print(quarantine_customers.head())

# Payments: refunds may be negative; any other negative amount is a sign error we fix and flag
pay = tables['fact_payments']
pay['is_refund'] = pay['payment_status'].eq('refunded')
sign_error = (pay['amount'] < 0) & ~pay['is_refund']
pay['amount_corrected'] = sign_error  # audit flag so every fix is traceable
pay.loc[sign_error, 'amount'] = pay.loc[sign_error, 'amount'].abs()

print(f"--- fact_payments | sign errors corrected: {sign_error.sum():,} ---")
print(pd.crosstab(pay['payment_status'], pay['amount'] < 0))  # negatives should only remain under 'refunded'

# Subscriptions: every reversed row is cancelled/churned, so the dates are swapped and flagged
subs = tables['fact_subscriptions']
reversed_mask = subs['end_date'].notna() & (subs['start_date'] > subs['end_date'])
subs['dates_swapped'] = reversed_mask  # audit flag so cohort analysis can exclude these rows if needed
# .values swaps by position; without it pandas would realign on column names and swap nothing
subs.loc[reversed_mask, ['start_date', 'end_date']] = subs.loc[reversed_mask, ['end_date', 'start_date']].values

print(f"--- fact_subscriptions | date pairs swapped: {reversed_mask.sum():,} ---")
print(pd.crosstab(subs['status'], subs['end_date'].isna()))  # active should sit only under end_date missing
print(subs['plan_id'].value_counts(dropna=False))

# Usage: login_count is a count and the negatives sit inside the normal 0-8 range, so the sign is flipped
usage = tables['fact_usage']
neg_login = usage['login_count'] < 0
usage['login_count_corrected'] = neg_login  # audit flag so every fix is traceable
usage.loc[neg_login, 'login_count'] = usage.loc[neg_login, 'login_count'].abs()

print(f"--- fact_usage | sign errors corrected: {neg_login.sum():,} of {len(usage):,} ---")
print(usage['login_count'].describe())  # min should now be 0 or above

# Verification printout: nulls and value ranges per table
for name, df in tables.items():
    print(f"--- {name} | shape: {df.shape} ---")
    print(df.isna().sum()[df.isna().sum() > 0])  # remaining nulls are intentional (dates, money)
    for col in [c for c in FINANCIAL_COLS + ['amount'] if c in df.columns]:
        print(f"{col} | min: {df[col].min()} | max: {df[col].max()} | nulls: {df[col].isna().sum():,}")

# Gate 1: only dim_customers may change row count, and only by what we dropped or quarantined
expected_rows = {name: len(df) for name, df in raw_tables.items()}
expected_rows['dim_customers'] -= exact_dropped + len(quarantine_customers)
for name, df in tables.items():
    assert len(df) == expected_rows[name], f"{name}: row count changed unexpectedly"

# Gate 2: plan money cannot be negative; payment amounts only when the row is a refund
for name, df in tables.items():
    for col in [c for c in FINANCIAL_COLS if c in df.columns]:
        neg = (df[col] < 0).sum()
        assert neg == 0, f"{name}.{col}: {neg:,} negative values"
assert pay.loc[pay['amount'] < 0, 'is_refund'].all(), "fact_payments: negative amount on a non-refund row"

# Gate 2b: counts cannot be negative after correction
assert (usage['login_count'] >= 0).all(), "fact_usage: negative login_count remains"

# Gate 3: start_date must occur before end_date (null end_date = still active)
if {'start_date', 'end_date'}.issubset(subs.columns):
    bad_dates = subs[subs['end_date'].notna() & (subs['start_date'] >= subs['end_date'])]
    print(f"Active subscriptions (no end_date): {subs['end_date'].isna().sum():,}")
    print(f"Date order violations: {len(bad_dates):,}")
    assert len(bad_dates) == 0, "start_date is not before end_date in some rows"
else:
    print(f"WARNING: date gate skipped, columns found: {list(subs.columns)}")  # never skip silently

# Gate 3b: status must agree with end_date (active = no end_date, everything else has one)
assert (subs['status'].eq('active') == subs['end_date'].isna()).all(), "status and end_date disagree"

# Gate 4: primary keys are unique and never filled with the placeholder label
for name, key in PRIMARY_KEYS.items():
    if key in tables[name].columns:
        assert tables[name][key].is_unique, f"{name}.{key}: duplicate keys"
        assert (tables[name][key] != 'Not Specified').all(), f"{name}.{key}: missing keys"

# Gate 5: foreign keys must point to existing parents
if 'customer_id' in subs.columns:
    orphans = ~subs['customer_id'].isin(tables['dim_customers']['customer_id'])
    print(f"Orphan subscriptions (customer): {orphans.sum():,}")
    assert orphans.sum() == 0, "Subscriptions reference unknown customers"
if 'plan_id' in subs.columns:
    orphan_plans = ~subs['plan_id'].isin(tables['dim_plans']['plan_id'])
    print(f"Orphan subscriptions (plan): {orphan_plans.sum():,}")
    assert orphan_plans.sum() == 0, "Subscriptions reference unknown plans"
if 'subscription_id' in pay.columns and 'subscription_id' in subs.columns:
    orphan_pay = ~pay['subscription_id'].isin(subs['subscription_id'])
    print(f"Orphan payments: {orphan_pay.sum():,}")
    assert orphan_pay.sum() == 0, "Payments reference unknown subscriptions"

print(f"Quality gates passed | tables: {len(tables)} | rows: {sum(len(d) for d in tables.values()):,}")

--- dim_customers | exact duplicates dropped: 50 | older records quarantined: 0 ---
Empty DataFrame
Columns: [customer_id, signup_date, country, industry, company_size, segment]
Index: []
--- fact_payments | sign errors corrected: 36 ---
amount          False  True 
payment_status              
failed            760      0
paid             1542      0
refunded          745     16
--- fact_subscriptions | date pairs swapped: 23 ---
end_date   False  True 
status                 
active         0   1043
cancelled    306      0
churned      151      0
plan_id
p_pro      749
p_ent      394
p_basic    357
Name: count, dtype: int64
--- fact_usage | sign errors corrected: 810 of 24,756 ---
count    24756.000000
mean         4.504968
std          2.292774
min          1.000000
25%          2.000000
50%          5.000000
75%          7.000000
max          8.000000
Name: login_count, dtype: float64
--- dim_customers | shape: (1500, 6) ---
Series([], dtype: int64)
--- dim_plans | shape: (3, 4) --

In [5]:
# ---- Cell 5 code ----
# Pre-load prep: MySQL-friendly flags, plus a look at the tables our gates have not covered yet
for name, df in tables.items():
    # MySQL has no True/False text type, so booleans become 1/0 (TINYINT)
    bool_cols = df.select_dtypes(include='bool').columns
    df[bool_cols] = df[bool_cols].astype(int)
    print(f"--- {name} | shape: {df.shape} ---")
    print(df.dtypes.to_dict())

# Needed to find the real price column and to design gates for usage and support
print(tables['dim_plans'])
print(tables['fact_usage'].head(3))
print(tables['fact_support'].head(3))

# Gate: no boolean columns may remain before export
for name, df in tables.items():
    assert df.select_dtypes(include='bool').shape[1] == 0, f"{name}: boolean columns remain"

print(f"Tables ready for export: {len(tables)} | rows: {sum(len(d) for d in tables.values()):,}")

--- dim_customers | shape: (1500, 6) ---
{'customer_id': dtype('O'), 'signup_date': dtype('<M8[ns]'), 'country': dtype('O'), 'industry': dtype('O'), 'company_size': dtype('O'), 'segment': dtype('O')}
--- dim_plans | shape: (3, 4) ---
{'plan_id': dtype('O'), 'plan_name': dtype('O'), 'monthly_price': dtype('float64'), 'billing_cycle': dtype('O')}
--- fact_subscriptions | shape: (1500, 7) ---
{'subscription_id': dtype('O'), 'customer_id': dtype('O'), 'plan_id': dtype('O'), 'start_date': dtype('<M8[ns]'), 'end_date': dtype('<M8[ns]'), 'status': dtype('O'), 'dates_swapped': dtype('int64')}
--- fact_payments | shape: (3063, 8) ---
{'payment_id': dtype('O'), 'subscription_id': dtype('O'), 'payment_date': dtype('<M8[ns]'), 'amount': dtype('float64'), 'payment_status': dtype('O'), 'payment_method': dtype('O'), 'is_refund': dtype('int64'), 'amount_corrected': dtype('int64')}
--- fact_usage | shape: (24756, 7) ---
{'usage_id': dtype('O'), 'customer_id': dtype('O'), 'event_date': dtype('<M8[ns]'),

In [6]:
# ---- Cell 6 code ----
# Target export: cleaned CSV per table, plus the quarantined customer rows for audit
export_tables = {**tables, 'quarantine_dim_customers': quarantine_customers}

for name, df in export_tables.items():
    # index=False keeps the pandas row numbers out of the CSV
    df.to_csv(DATA_DIR / f"cleaned_{name}.csv", index=False)

# Reload and compare so we know the saved files match memory
for name, df in export_tables.items():
    check = pd.read_csv(DATA_DIR / f"cleaned_{name}.csv")
    print(f"--- {name} | saved: {check.shape} ---")
    assert check.shape == df.shape, f"{name}: exported file shape differs"

print(f"Files exported: {len(export_tables)} | Total rows: {sum(len(d) for d in export_tables.values()):,}")

--- dim_customers | saved: (1500, 6) ---
--- dim_plans | saved: (3, 4) ---
--- fact_subscriptions | saved: (1500, 7) ---
--- fact_payments | saved: (3063, 8) ---
--- fact_usage | saved: (24756, 7) ---
--- fact_support | saved: (784, 7) ---
--- quarantine_dim_customers | saved: (0, 6) ---
Files exported: 7 | Total rows: 31,606


In [7]:
# ---- Cell 7 code ----
# Combined export: all cleaned tables plus the quarantine sheet in one Excel workbook
# Requires openpyxl (pip install openpyxl) if pandas cannot find an Excel writer
EXCEL_OUT = DATA_DIR / "Cleaned_SaaS_Analytics.xlsx"

# One sheet per table; Excel caps sheet names at 31 characters, so check before writing
for name in export_tables:
    assert len(name) <= 31, f"Sheet name too long for Excel: {name}"

with pd.ExcelWriter(EXCEL_OUT, engine='openpyxl') as writer:
    for name, df in export_tables.items():
        # index=False keeps the pandas row numbers out of the sheet
        df.to_excel(writer, sheet_name=name, index=False)
        print(f"--- {name} | written: {df.shape} ---")

# Reload and compare so we know the workbook matches memory
xl_check = pd.ExcelFile(EXCEL_OUT)
assert set(xl_check.sheet_names) == set(export_tables), "Sheet names differ from the exported tables"

for name, df in export_tables.items():
    check = xl_check.parse(name)
    print(f"--- {name} | reloaded: {check.shape} ---")
    assert check.shape == df.shape, f"{name}: workbook sheet shape differs"

print(f"Workbook saved: {EXCEL_OUT.name} | sheets: {len(xl_check.sheet_names)} | total rows: {sum(len(d) for d in export_tables.values()):,}")

--- dim_customers | written: (1500, 6) ---
--- dim_plans | written: (3, 4) ---
--- fact_subscriptions | written: (1500, 7) ---
--- fact_payments | written: (3063, 8) ---
--- fact_usage | written: (24756, 7) ---
--- fact_support | written: (784, 7) ---
--- quarantine_dim_customers | written: (0, 6) ---
--- dim_customers | reloaded: (1500, 6) ---
--- dim_plans | reloaded: (3, 4) ---
--- fact_subscriptions | reloaded: (1500, 7) ---
--- fact_payments | reloaded: (3063, 8) ---
--- fact_usage | reloaded: (24756, 7) ---
--- fact_support | reloaded: (784, 7) ---
--- quarantine_dim_customers | reloaded: (0, 6) ---
Workbook saved: Cleaned_SaaS_Analytics.xlsx | sheets: 7 | total rows: 31,606


In [8]:
# ---- Cell 8 code ----
# Pre-SQL verification: re-read the exported CSVs from disk, exactly as MySQL will see them
SQL_TABLES = ['dim_customers', 'dim_plans', 'fact_subscriptions', 'fact_payments', 'fact_usage', 'fact_support']
KEYS = {'dim_customers': 'customer_id', 'dim_plans': 'plan_id',
        'fact_subscriptions': 'subscription_id', 'fact_payments': 'payment_id'}

check_tables = {name: pd.read_csv(DATA_DIR / f"cleaned_{name}.csv") for name in SQL_TABLES}

for name, df in check_tables.items():
    print(f"--- {name} | shape: {df.shape} ---")
    print(df.isna().sum()[df.isna().sum() > 0])  # remaining nulls should only be end_date

    # Booleans must be 1/0 so they load into a TINYINT column
    assert df.select_dtypes(include='bool').shape[1] == 0, f"{name}: True/False columns remain, run Cell 4b before Cell 5"

    # Every date column must parse cleanly, or the SQL DATE load will fail
    for col in [c for c in df.columns if 'date' in c or 'timestamp' in c]:
        parsed = pd.to_datetime(df[col], errors='coerce')
        lost = df[col].notna().sum() - parsed.notna().sum()
        print(f"{col} | min: {parsed.min()} | max: {parsed.max()} | unparseable: {lost:,}")
        assert lost == 0, f"{name}.{col}: {lost:,} unparseable dates"

    # No exact duplicate rows should reach the database
    print(f"Duplicate rows: {df.duplicated().sum():,}")

# Price rule: plan money columns must exist and cannot be negative
money_cols = [(n, c) for n, d in check_tables.items() for c in d.columns if any(k in c for k in ['price', 'mrr'])]
if not money_cols:
    print(f"WARNING: no price/mrr column found, dim_plans columns: {list(check_tables['dim_plans'].columns)}")  # never skip silently
for name, col in money_cols:
    print(f"{name}.{col} | min: {check_tables[name][col].min()} | max: {check_tables[name][col].max()}")
    assert (check_tables[name][col] >= 0).all(), f"{name}.{col}: negative values"

# Usage and support: counts, durations and times cannot be negative
for name in ['fact_usage', 'fact_support']:
    num = check_tables[name].select_dtypes(include='number')
    neg = {c: int((num[c] < 0).sum()) for c in num.columns if (num[c] < 0).any()}
    print(f"{name} negative values: {neg if neg else 'None'}")
    assert not neg, f"{name}: negative values in {list(neg)}"

# Keys unique in every table that has one
for name, key in KEYS.items():
    assert check_tables[name][key].is_unique, f"{name}.{key}: duplicate keys"

print(f"Ready for SQL load | tables: {len(check_tables)} | rows: {sum(len(d) for d in check_tables.values()):,}")

--- dim_customers | shape: (1500, 6) ---
Series([], dtype: int64)
signup_date | min: 2025-01-01 00:00:00 | max: 2025-12-31 00:00:00 | unparseable: 0
Duplicate rows: 0
--- dim_plans | shape: (3, 4) ---
Series([], dtype: int64)
Duplicate rows: 0
--- fact_subscriptions | shape: (1500, 7) ---
end_date    1043
dtype: int64
start_date | min: 2024-12-16 00:00:00 | max: 2026-01-01 00:00:00 | unparseable: 0
end_date | min: 2025-01-03 00:00:00 | max: 2026-07-30 00:00:00 | unparseable: 0
dates_swapped | min: 1970-01-01 00:00:00 | max: 1970-01-01 00:00:00.000000001 | unparseable: 0
Duplicate rows: 0
--- fact_payments | shape: (3063, 8) ---
Series([], dtype: int64)
payment_date | min: 2025-01-03 00:00:00 | max: 2026-03-31 00:00:00 | unparseable: 0
Duplicate rows: 0
--- fact_usage | shape: (24756, 7) ---
Series([], dtype: int64)
event_date | min: 2025-01-01 00:00:00 | max: 2025-12-17 00:00:00 | unparseable: 0
Duplicate rows: 0
--- fact_support | shape: (784, 7) ---
Series([], dtype: int64)
ticket_da